# 01 — Dataset Audit & Inventory
## RoadSafe Delhi: Explainable Road Risk Mapping & Safety Prioritization

### Objectives
1. Perform programmatic inventory of all files in `Datasets/`.
2. Inspect file schemas, nulls, and date coverages.
3. Validate multi-year Meteostat records (2022–2025).
4. Quarantine synthetic `risk_score` to prevent target leakage.


In [ ]:
import os
import sys

# Robust project-relative path configuration
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
DATASETS_DIR = os.path.join(PROJECT_ROOT, "Datasets")
DATA_PROC_DIR = os.path.join(PROJECT_ROOT, "data_processed")
DATA_ENG_DIR = os.path.join(PROJECT_ROOT, "data_engineered")
DATA_FINAL_DIR = os.path.join(PROJECT_ROOT, "data_final")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
EDA_DIR = os.path.join(OUTPUTS_DIR, "eda")
MAPS_DIR = os.path.join(OUTPUTS_DIR, "maps")
TABLES_DIR = os.path.join(OUTPUTS_DIR, "tables")
REPORTS_DIR = os.path.join(OUTPUTS_DIR, "reports")

print("Project Root:", PROJECT_ROOT)


### 1. Data Inventory Table


In [ ]:
import pandas as pd
df_inv = pd.read_csv(os.path.join(TABLES_DIR, "data_inventory.csv"))
df_inv


### 2. Multi-Year Meteostat Inspection


In [ ]:
import gzip

meteo_dir = os.path.join(DATASETS_DIR, "Weather Datasets")
meteo_files = [f for f in os.listdir(meteo_dir) if f.startswith("42182") and f.endswith(".gz")]

for mf in sorted(meteo_files):
    fp = os.path.join(meteo_dir, mf)
    df_m = pd.read_csv(fp, compression='gzip')
    print(f"File: {mf:<20} | Rows: {len(df_m):<6} | Years: {df_m['year'].unique()}")


### 3. Leakage Flag on Crash Dataset


In [ ]:
import zipfile
acc_zip_p = os.path.join(DATASETS_DIR, "Indian-Road-Accident-Dataset-Analysis-2022-2025-Python---Jupyternotebook-main.zip")
with zipfile.ZipFile(acc_zip_p) as z:
    with z.open('Indian-Road-Accident-Dataset-Analysis-2022-2025-Python---Jupyternotebook-main/indian_roads_dataset.csv') as f:
        df_acc = pd.read_csv(f)

print(f"Total National Records: {len(df_acc):,}")
delhi_acc = df_acc[df_acc['state'].astype(str).str.contains('Delhi', case=False, na=False)]
print(f"Delhi Crash Records: {len(delhi_acc):,}")
assert 'risk_score' in delhi_acc.columns
print("QUARANTINE ACTION: 'risk_score' identified and permanently removed from analysis pipeline.")
